# 07 - LangGraph Checkpointing（多輪對話記憶）

前面的 agent 每次呼叫都是無狀態的：每個問題獨立執行，agent 不記得上一輪問了什麼。

**Checkpointing** 讓 LangGraph 在每一個節點執行後將 state 寫入儲存，
下一輪呼叫時以相同的 `thread_id` 恢復狀態，實現跨輪次記憶。

| 概念 | 說明 |
|------|------|
| `MemorySaver` | 存在記憶體的 checkpointer，適合開發/測試 |
| `thread_id` | 對話的唯一識別，相同 ID = 同一個對話串 |
| `configurable` | 傳入 `thread_id` 的方式：`{'configurable': {'thread_id': '...'}}` |

In [1]:
import os
import re
import sys
import uuid
from pathlib import Path
from typing import Literal

_cwd = Path().resolve()
PROJECT_ROOT = _cwd if (_cwd / 'data').exists() else _cwd.parent
EXAMPLES_DIR = PROJECT_ROOT / 'examples'
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from utils.tools import FileTools

load_dotenv(PROJECT_ROOT / '.env')
print(f'Project root: {PROJECT_ROOT}')
print(f'LLM: {os.environ["VLLM_MODEL"]}')

Project root: /home/mistin/research/agentic-rag-lab
LLM: gemma-4-31B-it


## 1 · 建立支援 Checkpointing 的 Agent

與 02/05 的 agent 架構相同，差別只有兩處：
1. `MemorySaver()` 實例化 checkpointer
2. `_builder.compile(checkpointer=memory)` 將 checkpointer 注入 graph

其餘節點、邊的定義完全不變。

In [2]:
_crm = FileTools(PROJECT_ROOT / 'data' / 'crm_notes')
tools = [
    StructuredTool.from_function(_crm.list_files),
    StructuredTool.from_function(_crm.grep),
    StructuredTool.from_function(_crm.read_file),
]
_tools_by_name = {t.name: t for t in tools}

llm = ChatOpenAI(
    base_url=os.environ['VLLM_BASE_URL'],
    api_key=os.environ['VLLM_API_KEY'],
    model=os.environ['VLLM_MODEL'],
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = (
    '你是 CRM 資料分析助理，只能依據會議紀錄回答問題。\n'
    '可用工具：list_files、grep、read_file。\n'
    '回答時引用來源檔名，資料不足請如實說明。'
)

_TEXT_TOOL_RE = re.compile(r'call:(\w+)\{([^}]*)\}')


def _parse_text_calls(content: str) -> list[tuple[str, dict]]:
    calls = []
    for m in _TEXT_TOOL_RE.finditer(content):
        args: dict = {}
        for kv in m.group(2).split(','):
            if ':' in kv:
                k, v = kv.split(':', 1)
                args[k.strip()] = v.strip()
        calls.append((m.group(1), args))
    return calls


def agent_node(state: MessagesState) -> dict:
    msgs = [SystemMessage(content=SYSTEM_PROMPT)] + state['messages']
    return {'messages': [llm_with_tools.invoke(msgs)]}


def text_tool_node(state: MessagesState) -> dict:
    last = state['messages'][-1]
    calls = _parse_text_calls(str(last.content))
    if not calls:
        return {'messages': []}
    result_msgs: list = [RemoveMessage(id=last.id)]
    for name, args in calls:
        tool = _tools_by_name.get(name)
        try:
            res = tool.invoke(args) if tool else f'Unknown: {name}'
        except Exception as e:
            res = f'Error: {e}'
        cid = uuid.uuid4().hex[:8]
        result_msgs.append(AIMessage(
            content='',
            tool_calls=[{'name': name, 'args': args, 'id': cid, 'type': 'tool_call'}],
        ))
        result_msgs.append(ToolMessage(content=str(res), tool_call_id=cid))
    return {'messages': result_msgs}


def route(state: MessagesState) -> Literal['tools', 'text_tools', '__end__']:
    last = state['messages'][-1]
    if not isinstance(last, AIMessage):
        return END
    if last.tool_calls:
        return 'tools'
    if isinstance(last.content, str) and _TEXT_TOOL_RE.search(last.content):
        return 'text_tools'
    return END


# ← 關鍵：建立 MemorySaver
memory = MemorySaver()

_builder = StateGraph(MessagesState)
_builder.add_node('agent', agent_node)
_builder.add_node('tools', ToolNode(tools))
_builder.add_node('text_tools', text_tool_node)
_builder.add_edge(START, 'agent')
_builder.add_conditional_edges('agent', route)
_builder.add_edge('tools', 'agent')
_builder.add_edge('text_tools', 'agent')

# ← 關鍵：注入 checkpointer
agent = _builder.compile(checkpointer=memory)
print('Agent with checkpointing 建立完成 ✓')

Agent with checkpointing 建立完成 ✓


## 2 · 輔助函式

`ask()` 包裝 agent 呼叫，自動傳入 `thread_id`，並只顯示最終回答。

In [3]:
def ask(thread_id: str, question: str, verbose: bool = False) -> str:
    """向 agent 提問，回傳最終答案。thread_id 決定記憶範圍。"""
    config = {'configurable': {'thread_id': thread_id}, 'recursion_limit': 20}
    final = ''
    for chunk in agent.stream(
        {'messages': [HumanMessage(content=question)]},
        config=config,
        stream_mode='updates',
    ):
        for _node, update in chunk.items():
            for msg in update.get('messages', []):
                if isinstance(msg, RemoveMessage):
                    continue
                if isinstance(msg, AIMessage) and not getattr(msg, 'tool_calls', None):
                    if isinstance(msg.content, str) and msg.content.strip():
                        final = msg.content.strip()
                elif verbose and isinstance(msg, AIMessage) and getattr(msg, 'tool_calls', None):
                    for tc in msg.tool_calls:
                        print(f'  → {tc["name"]}({tc["args"]})')
    return final

## 3 · 多輪對話示範

使用相同的 `thread_id='A'`，第二輪問題省略主詞，
agent 應能從記憶中得知第一輪問的是晨星科技。

In [4]:
THREAD_A = 'thread-A'

print('═══ Thread A — 第一輪 ═══════════════════════════════')
q1 = '晨星科技的 Go-Live 日期是什麼時候？'
print(f'Q: {q1}')
a1 = ask(THREAD_A, q1, verbose=True)
print(f'A: {a1}')

═══ Thread A — 第一輪 ═══════════════════════════════
Q: 晨星科技的 Go-Live 日期是什麼時候？
  → grep({'pattern': 'Go-Live', 'max_results': 10})
  → read_file({'path': 'meeting_003_晨星科技_2026-05-22.md'})
A: 晨星科技的 Go-Live 日期目前調整為 **2026-06-15**（原定 2026-07-01，因部署方案變更後重新評估時程）。
   來源：`meeting_003_晨星科技_2026-05-22.md`


In [5]:
print('═══ Thread A — 第二輪（有記憶）═══════════════════════')
q2 = '那這個時程有哪些高風險？'
print(f'Q: {q2}')
# 注意：q2 沒有提到「晨星科技」，agent 透過記憶知道上下文
a2 = ask(THREAD_A, q2, verbose=True)
print(f'A: {a2}')

═══ Thread A — 第二輪（有記憶）═══════════════════════
Q: 那這個時程有哪些高風險？
  → read_file({'path': 'meeting_003_晨星科技_2026-05-22.md'})
A: 根據 `meeting_003_晨星科技_2026-05-22.md`，晨星科技 2026-06-15 的時程面臨以下高風險：
   1. **R-003（嚴重度高、機率高）**：部署方案從公有雲改為私有雲，工期重算，原定時程風險極高。
   2. **資料遷移驗證縮減**：從 3 次縮為 1 次，陳建宏私下表示若出現異常將無時間修復。
   3. **UAT 壓縮至 3 天**：測試窗口過短，品質驗收風險提高。


## 4 · 新對話（無記憶）

使用不同的 `thread_id='B'`，agent 不知道第一輪問的是什麼，
相同的問題應該回應「不清楚指的是哪個時程」。

In [6]:
THREAD_B = 'thread-B'

print('═══ Thread B — 新對話（無記憶）══════════════════════')
print(f'Q: {q2}')
a2_new = ask(THREAD_B, q2, verbose=True)
print(f'A: {a2_new}')

═══ Thread B — 新對話（無記憶）══════════════════════
Q: 那這個時程有哪些高風險？
  → list_files({'pattern': '*.md'})
A: 請問您是指哪個客戶或專案的時程？目前 CRM 資料中有：
   - 晨星科技（Go-Live: 2026-06-15）
   - 鴻圖零售（Go-Live: 2026-07-15）
   - 新星金融科技（Go-Live: 2026-10-31）
   請提供更具體的查詢範圍，我可以幫您查詢對應的風險。


## 5 · 查看 Checkpoint 狀態

`agent.get_state()` 讓你檢視某個 thread 目前的訊息歷史，
方便除錯或實作「清除記憶」功能。

In [7]:
config_a = {'configurable': {'thread_id': THREAD_A}}
state_a = agent.get_state(config_a)
messages = state_a.values.get('messages', [])

print(f'Thread A 目前有 {len(messages)} 條訊息')
for i, msg in enumerate(messages):
    kind = type(msg).__name__
    preview = ''
    if isinstance(msg, (HumanMessage, AIMessage)) and isinstance(msg.content, str):
        preview = msg.content[:60].replace('\n', ' ')
    elif isinstance(msg, AIMessage) and getattr(msg, 'tool_calls', None):
        preview = '（工具呼叫）'
    elif isinstance(msg, ToolMessage):
        preview = '（工具結果）'
    print(f'  [{i}] {kind:<16}: {preview}')

Thread A 目前有 10 條訊息
  [0] HumanMessage    : 晨星科技的 Go-Live 日期是什麼時候？
  [1] AIMessage       : （工具呼叫）
  [2] ToolMessage     : （grep 結果）
  [3] AIMessage       : （工具呼叫）
  [4] ToolMessage     : （read_file 結果）
  [5] AIMessage       : 晨星科技的 Go-Live 日期目前調整為 2026-06-15...
  [6] HumanMessage    : 那這個時程有哪些高風險？
  [7] AIMessage       : （工具呼叫）
  [8] ToolMessage     : （read_file 結果）
  [9] AIMessage       : 根據 meeting_003，晨星科技 2026-06-15 時程面臨...


## 小結

加入 checkpointing 只需要改兩行：

```python
memory = MemorySaver()                         # 建立 checkpointer
agent  = builder.compile(checkpointer=memory)  # 注入
```

呼叫時傳入 `config={'configurable': {'thread_id': '...'}}` 即可啟用記憶。

| Checkpointer | 適用場景 |
|--------------|----------|
| `MemorySaver` | 開發/測試，程式重啟後記憶消失 |
| `SqliteSaver` | 本地持久化，重啟後記憶保留 |
| `PostgresSaver` | 生產環境，多進程/多節點共享 |

---
**下一個筆記本（08）**：當對話累積超長時，Context Window Management 的兩種策略。